In [ ]:
# import os
# os.environ['JAX_PLATFORMS'] = 'cpu'

import os
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ["XLA_FLAGS"] = f"--xla_gpu_cuda_data_dir={os.environ.get('CUDA_HOME', '')}"

# This is required to run multiple processes with JAX.
from multiprocessing import set_start_method
set_start_method('forkserver', force=True)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
from tqdm import tqdm
from pathlib import Path

# print(jax.devices())
# jax.config.update("jax_debug_nans", True)

from train import Trainer
from config import Config, AssimConfig
import data

In [ ]:
run_dir = Path("/nas/cee-water/cjgleason/ted/swot-ml/runs/sf_grnn_pooled/swot_20260504_193818")
trainer = Trainer.load_from_run_or_state(run_dir)

In [ ]:
assim_path = Path("/nas/cee-water/cjgleason/ted/swot-ml/runs/_assim/swot.yml")
assim_cfg = AssimConfig.from_file(assim_path)
trainer.cfg.add_assimilation(assim_cfg)
trainer.cfg.quiet = False
trainer.cfg.log = False
trainer.cfg.num_workers = 0
cfg = trainer.cfg

dls = data.get_dataloaders(cfg)
trainer.training_dl = dls['train']

new_model = trainer.model
for new_group in assim_cfg.new_features.keys():
    n_group_features = len(dataset.features.dynamic[new_group])
    group_network_sizes = assim_cfg.model_args.get(new_group, {})
    new_model.add_assimilator(new_group, n_group_features, group_network_sizes)
    trainer.cfg.model_args.sparse_sizes[new_group] = n_group_features
trainer.replace_model(new_model)

if assim_cfg.freeze_base:
    if getattr(new_model, 'backbone_leaves', None):
        trainer.freeze_components(new_model.backbone_leaves)
    elif 'get_backbone' in dir(new_model):
        trainer.freeze_components(new_model.get_backbone())
    else:
        print("Assim was config'd to freeze base model but base not defined.")

In [ ]:
trainer.model.get_backbone()

In [ ]:
import equinox as eqx

diff_model, static_model = eqx.partition(trainer.model, trainer.filter_spec)



In [ ]:
sizes = {'test':1, 'test2':10}
x\

In [ ]:
for basins, subbasins, dates, batch in tqdm(dataloader):
    break


In [ ]:
import equinox as eqx

model = eqx.tree_at(lambda m: m.return_weights, trainer.model, True)

In [ ]:
key = jax.random.PRNGKey(0)
y_hat, weights = model(batch, key)

In [ ]:
np.where(~np.isnan(batch.dynamic['swot-river'][:,0,0]))

In [ ]:
np.where(~np.isnan(weights['fusion']['weights'][:,0,0]))

In [ ]:
batch.dynamic['swot-river'][4,0,:]

In [ ]:
weights['fusion']['weights'][4,0,:]

In [ ]:
weights['fusion']['weights'].shape

In [ ]:
y_true = batch.y['discharge'][0,...,0]
node_mask = batch.node_mask

valid_mask = (~np.isnan(y_true)) & node_mask
masked_y = np.where(valid_mask, y_true, 0)
expanded_mask = np.expand_dims(valid_mask, axis=-1)

expanded_mask.shape

In [ ]:
np.sum(valid_mask)

In [ ]:
y_hat_target = y_hat['discharge']
masked_y_hat = {
    "mu": jnp.where(expanded_mask, y_hat_target["mu"], 0.0),
    "sigma": jnp.where(expanded_mask, y_hat_target["sigma"], 1.0),  # sigma=1 is safe
    "log_pi": jnp.where(expanded_mask, y_hat_target["log_pi"], 1.0 / y_hat_target["log_pi"].shape[-1]),  # uniform is safe
}

In [ ]:
y_obs = jnp.expand_dims(masked_y, axis=-1)  # Allow broadcasting to each error model
mu = masked_y_hat["mu"]
sigma = masked_y_hat["sigma"]
log_pi = masked_y_hat["log_pi"]

# 1. log(pdf) = -log(sigma) - 0.5*log(2*pi) - 0.5 * ((y-mu)/sigma)^2
log_sigma = jnp.log(sigma)
log_2pi = jnp.log(2.0 * jnp.pi)

# Calculate Mahalanobis distance with clipping
# Gradient of sq_diff w.r.t sigma is proportional to 1/sigma^3. 
# Clipping here is the 'safety valve' for your grad norm.
scaled_diff = (y_obs - mu) / sigma
sq_diff = jnp.square(jnp.clip(scaled_diff, -10.0, 10.0))

log_phi = -log_sigma - 0.5 * log_2pi - 0.5 * sq_diff

# 2. log(sum(pi * pdf)) = logsumexp(log_pi + log_phi)
# Both terms are already in log-space.
log_likelihood = jax.scipy.special.logsumexp(log_pi + log_phi, axis=-1)

# 3. Masked Mean
# Note: sum(mask) + 1e-8 prevents div-by-zero on empty batches
-jnp.sum(log_likelihood * valid_mask) / (jnp.sum(valid_mask) + 1e-8)

In [ ]:
log_pi[valid_mask]

In [ ]:
np.mean(log_weighted_likelihoods, where=valid_mask)

In [ ]:
valid_mask.shape

In [ ]:
y_hat_target['sigma'].shape

In [ ]:
y_hat['sigma'][valid_mask]

In [ ]:
masked_y_hat['sigma'][valid_mask[0,...], :]

In [ ]:
node_mask[jnp.newaxis,:].shape

In [ ]:
log_pi

In [ ]:
-jnp.mean(log_weighted_likelihoods, where=valid_mask)

In [ ]:
valid_mask.shape

In [ ]:
plt.plot(jnp.isnan(log_weighted_likelihoods).T)

In [ ]:
valid_mask.shape

In [ ]:
log_weighted_likelihoods.shape

In [ ]:
import train

train.loss.gmm_nll(masked_y, masked_y_hat, valid_mask, None)

In [ ]:
np.nansum(batch.y['discharge'][0,:,0])

In [ ]:
obs_emb['swot-river'][0].shape

In [ ]:
batch.dynamic['swot-river'].shape

In [ ]:
sub_m

In [ ]:
batch.dynamic['swot-river'][:, batch.node_mask, :]

In [ ]:
batch.node_mask[None,:,None].shape